# JalPulse -- EPBL / BGCC Case Consilium 2026

Runs the full pipeline interactively: fetch raw sources -> extract structured CSVs -> compute the JalPulse score -> summarise into reports. Every stage writes its output to disk (`data/raw/`, `data/processed/`, `reports/`) so you can also just run `python run_all.py` from the terminal and skip this notebook entirely -- this notebook exists for inspecting intermediate results and eyeballing the data as you go.

All weights, thresholds and source URLs live in `config.py` -- change assumptions there, not in this notebook.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

import config
from src import fetch, extract, score, report

## Stage 1 -- Fetch raw sources
Downloads to `data/raw/sources/`. Cached: re-running this cell is a no-op unless a file is missing or you pass `force=True`.

In [ ]:
fetch_results = fetch.fetch_all(force=False)
fetch_results

## Stage 2 -- Extract structured raw CSVs
Each source becomes one CSV in `data/raw/`. Nothing is scored or weighted yet -- this is faithful, traceable extraction only.

In [ ]:
extracted = extract.extract_all(force=False)
goa_hotels_raw = extracted["goa_hotels"]
goa_hotels_raw.head()

In [ ]:
print(f"Rows parsed: {len(goa_hotels_raw):,}")
print(f"Registry max serial: {goa_hotels_raw['sr'].max():,}")
print(f"Coverage vs. max serial: {len(goa_hotels_raw)/goa_hotels_raw['sr'].max():.1%}")
print(f"Low-confidence name extractions: {(goa_hotels_raw['name_confidence']=='low').sum()} "
      f"({(goa_hotels_raw['name_confidence']=='low').mean():.1%}) -- verify these against the source PDF before quoting a name.")

In [ ]:
extracted["epbl_facts"]

In [ ]:
extracted["drdo_licence"]

In [ ]:
extracted["gspcb_rules"]

In [ ]:
extracted["rera_mis"]

## Stage 3 -- Score: the JalPulse engine
Three signal layers (commercial value, regulatory urgency, serviceability) -> `value_score`. A decision-fit discount (does this property match EPBL's owner/GM-approved persona, not a corporate-procured chain) -> `phase1_score`. See `config.py` for every weight/threshold, and `src/score.py` docstring for the reasoning.

In [ ]:
results = score.run()
scored = results["scored"]
phase1 = results["phase1"]
taluka_summary = results["taluka_summary"]

In [ ]:
print("Tier distribution:")
print(scored["tier"].value_counts())
print()
print("Tier distribution (%):")
print((scored["tier"].value_counts(normalize=True) * 100).round(1))

In [ ]:
priority = scored[scored["tier"] == "Priority"]
print(f"Priority tier: {len(priority)} properties")
priority["taluka"].value_counts().head(10)

### The value-vs-winnability check
Raw `value_score` alone over-indexes on large room counts, which pulls in branded/chain-managed properties that don't match EPBL's actual buyer persona (independent owner/GM, direct approval). Compare the unfiltered top 15 to the fit-filtered Phase-1 shortlist below.

In [ ]:
top100 = scored.nlargest(100, "value_score")
big_in_top100 = (top100["rooms"] > config.FIT_SWEET_SPOT_MAX_ROOMS).sum()
print(f"{big_in_top100} of the top 100 properties by raw value_score are above "
      f"{config.FIT_SWEET_SPOT_MAX_ROOMS} rooms (likely chain-managed, low near-term fit).")
print()
print("Top 15 by raw value_score (unfiltered):")
scored.nlargest(15, "value_score")[["name", "taluka", "rooms", "value_score", "decision_fit", "tier"]]

In [ ]:
print(f"Phase-1 fit pool ({config.FIT_SWEET_SPOT_MIN_ROOMS}-{config.FIT_SWEET_SPOT_MAX_ROOMS} rooms): {len(phase1)} properties")
print()
print("Top 15 Phase-1 shortlist (value-ranked within the owner-decision fit band):")
phase1.nlargest(15, "value_score")[["name", "taluka", "rooms", "value_score", "phase1_score"]]

In [ ]:
taluka_summary.head(10)

### Optional: quick chart
Requires `matplotlib` (`pip install matplotlib`); skipped gracefully if not installed. Not required for the CSV/report outputs.

In [ ]:
try:
    import matplotlib.pyplot as plt
    top_talukas = taluka_summary.head(8).set_index("taluka")[["Priority", "Watch", "Long-tail"]]
    top_talukas.plot(kind="bar", stacked=True, figsize=(9, 5), title="JalPulse tier by taluka (top 8)")
    plt.ylabel("Properties")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed -- skipping chart. `pip install matplotlib` to enable.")

## Stage 4 -- Reports
Writes `reports/jalpulse_summary.md` (findings, ready to lift into slide notes) and `reports/evidence_log.md` (every disclosed/extracted fact, tagged [P]/[CD]/[D]/[A], for footnoting the deck).

In [ ]:
summary_path = report.generate_jalpulse_summary()
evidence_path = report.generate_evidence_log()
print(f"Wrote: {summary_path}")
print(f"Wrote: {evidence_path}")

In [ ]:
print(summary_path.read_text()[:3000])